In [6]:
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from signal_insight import config, quality, viz, geo


con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

cell = pd.read_sql("SELECT * FROM ods_cell", con)
sites = cell[['lng', 'lat']].drop_duplicates().reset_index(drop=True)
# 给站点做编号：编号补0到S00005
sites['site_id'] = "S" + (sites.index + 1).astype(str).str.zfill(5)

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")
sites['districts'] = geo.point_in_polygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")


# 没坐标点扇区定位不了
cell = cell.dropna(subset=['lng','lat']).copy()
# 不重复的坐标=一个站点
# 把站点号贴回每个扇区
cell = cell.merge(sites, on=['lng', 'lat'],how='left')

In [7]:
cell = pd.read_sql("SELECT * FROM ods_cell", con)
sites = cell[['lng', 'lat']].drop_duplicates().reset_index(drop=True)
# 给站点做编号：编号补0到S00005
sites['site_id'] = "S" + (sites.index + 1).astype(str).str.zfill(5)

districts = geo.rings_from_points(
    pd.read_sql("SELECT * FROM ods_district_boundary", con), "district")
sites['districts'] = geo.point_in_polygons(sites['lng'], sites['lat'], districts)
sites['districts'] = sites['districts'].fillna("市外")


# 没坐标点扇区定位不了
cell = cell.dropna(subset=['lng','lat']).copy()
# 不重复的坐标=一个站点
# 把站点号贴回每个扇区
cell = cell.merge(sites, on=['lng', 'lat'],how='left')

In [ ]:
# 读取dwd表
dwd = pd.read_sql("SELECT * FROM dwd_signal", con, parse_dates=['ts'])
# 两个表格合并
joined = dwd.merge(cell[['lac_cell', "site_id", 'lng', 'lat']],
                on='lac_cell',
                how='left')
out_share = (sites['districts'] == '市外').mean()
rec_district = joined.merge(sites[['site_id', 'districts']],
            on="site_id",how="left")
rec_out = (rec_district['districts'] =='市外').mean()

v2_pts = pd.read_sql("SELECT * FROM ods_aoi_v2", con)
aoi_v2 = geo.rings_from_points(v2_pts,"name")

sites['aoi'] = geo.point_in_polygons(sites['lng'],sites['lat'], aoi_v2)
dim_cell = cell.merge(sites[['site_id', 'aoi']],
                      on='site_id',
                      how='left')
def split_at_jump(values: pd.Series) -> tuple[list[str], list[str], float]:
    """按数值从小到大排，找相邻两项比值最大的地方切开。
    返回 (断崖之前的名字, 断崖之后的名字, 断崖处的倍数)。D3_02、D3_03 用它决定强调哪一组。"""
    v = values.sort_values()
    r = v.values[1:] / v.values[:-1]
    k = int(r.argmax())
    return list(v.index[: k + 1]), list(v.index[k + 1:]), float(r[k])

site = dim_cell.groupby('site_id').agg(
    lng=("lng", 'first'),
    lat=('lat', 'first'),
    district=('districts', 'first')
)